# In-world figures

Example of generating images and showing them in VR:

<video controls src="./assets/in_world_figures.webm">

## Setup runner & utilities

In [1]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/17-ala.xml")
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: in-world figures")
imd_runner.load(0)

In [2]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)
utilities.use_transform_handles()

In [9]:
# this will be integrated into nanover later, for now it doesn't work with recordings
RESOURCE_PATHS: dict[str, str] = {}


def add_resource(key: str, path: str):
    RESOURCE_PATHS[key] = path

    # recorders can use this to fetch resources without searching scene for usage
    utilities.set_shared_state_value("resources.ids", list(RESOURCE_PATHS.keys()))


def fetch_resource(key: str):
    path = RESOURCE_PATHS.get(key, None)

    if path is None:
        return {}

    with open(path, "rb") as file:
        return dict(
            data=file.read(),
        )


# xr client knows to use this command to retrieve image data
utilities.define_command("resources/fetch", handler=fetch_resource)

## Graph generation and display

Create a moveable handle for the figure:

In [8]:
from nanover.utilities.transforms import Transform

utilities.transforms.update_transform("graph", transform=Transform.identity(), parent="simulation")
utilities.handles.update_handle("graph", parent="graph")

Add a command to generate a graph image and update it in the scene:

In [5]:
from uuid import uuid4
import matplotlib.pyplot as plt


def display_graph():
    positions = imd_runner.app_server.frame_publisher.current_frame.particle_positions
    x, y, z = positions[:, 0], positions[:, 1], positions[:, 2]

    plt.scatter(x, y)
    plt.savefig("graph.png")
    plt.close()

    texture_id = f"graph.{uuid4()}"
    add_resource(texture_id, "graph.png")
    utilities.objects.update_sprite("graph", texture=texture_id, parent="graph", size=5, color=[1, 1, 1, .5])


utilities.define_command("user/graph", handler=display_graph, icon="📊", label="generate graph")